# Knowledge Insulation (KI) — recipe walkthrough

Manual demonstration that the KI recipe (May 2025 paper) does what it claims:

1. **Stop-gradient at the VLM ↔ action-expert interface** prevents the expert's loss from updating VLM weights.
2. The **discrete-token loss** on the VLM still trains the VLM normally.
3. The combined `ki_loss` produces a single scalar that can be `.backward()`'d.

We use toy stand-ins for the VLM and action expert. The point is to make the *recipe* legible, not to train a real policy.

Paper: `papers/2025-05-28_ki_train-fast-run-fast-generalize-better.pdf`

In [ ]:
import copy
import torch
from torch import nn
import matplotlib.pyplot as plt

from pi_stack.training.ki import (
    KIConfig,
    flow_matching_target,
    insulate,
    ki_loss,
)

torch.manual_seed(0)
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print('device:', device)

## Toy backbone + action expert

- **Backbone**: token-embedding + linear head → returns `(logits, features)`.
- **Action expert**: pools VLM features, concatenates with `(a_t, t)`, projects to `D`.

Both are tiny — the gradient-flow demonstration is what matters.

In [ ]:
VOCAB, FEAT = 32, 16
B, T, H, D = 8, 12, 20, 7

class TinyVLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, FEAT)
        self.transformer = nn.TransformerEncoderLayer(d_model=FEAT, nhead=4, batch_first=True)
        self.head = nn.Linear(FEAT, VOCAB)
    def forward(self, ids):
        h = self.transformer(self.embed(ids))
        return self.head(h), h

class TinyExpert(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(FEAT + D + 1, D)
    def forward(self, a_t, t, ctx):
        pooled = ctx.mean(dim=1, keepdim=True).expand(-1, a_t.size(1), -1)
        t_feat = t.view(-1, 1, 1).expand(-1, a_t.size(1), 1)
        return self.proj(torch.cat([a_t, t_feat, pooled], dim=-1))

vlm = TinyVLM().to(device)
expert = TinyExpert().to(device)

for name, m in [('vlm', vlm), ('expert', expert)]:
    n = sum(p.numel() for p in m.parameters())
    print(f'{name:6s}  {n:>6,d} params')

## A. Gradient-flow check

Forward both nets, compute *only* the action-expert (continuous) loss, backward. With KI in place the VLM must have zero gradient; with KI off it must have non-zero gradient.

In [ ]:
def expert_only_grad_norm(insulation_on: bool) -> tuple[float, float]:
    """Returns (vlm_grad_norm, expert_grad_norm) after backward."""
    vlm.zero_grad(set_to_none=True)
    expert.zero_grad(set_to_none=True)
    ids = torch.randint(0, VOCAB, (B, T), device=device)
    actions = torch.randn(B, H, D, device=device)
    noise = torch.randn_like(actions)
    t = torch.rand(B, device=device)
    _logits, feats = vlm(ids)
    ctx = insulate(feats, enabled=insulation_on)
    a_t, target = flow_matching_target(actions, noise, t)
    v_pred = expert(a_t, t, ctx)
    loss = torch.nn.functional.mse_loss(v_pred, target)
    loss.backward()
    vlm_g = sum((p.grad.norm()**2).item() for p in vlm.parameters() if p.grad is not None) ** 0.5
    exp_g = sum((p.grad.norm()**2).item() for p in expert.parameters() if p.grad is not None) ** 0.5
    return vlm_g, exp_g

for on in (True, False):
    vg, eg = expert_only_grad_norm(on)
    print(f'insulation={str(on):5s}  vlm_grad_norm={vg:.4e}  expert_grad_norm={eg:.4e}')

Expected: with insulation on the VLM grad is **exactly zero**; with insulation off it's non-zero.

## B. Full KI step — does the VLM still train via the discrete loss?

Run 200 training steps with the full dual loss. Track:
- discrete loss (next-token CE through the VLM)
- continuous loss (flow-matching MSE on the action expert)
- L2 distance between the *current* and *initial* VLM weights — confirms the VLM is still moving

In [ ]:
torch.manual_seed(0)
vlm = TinyVLM().to(device)
expert = TinyExpert().to(device)
vlm_init = copy.deepcopy(vlm).requires_grad_(False)

opt = torch.optim.Adam(list(vlm.parameters()) + list(expert.parameters()), lr=3e-4)
history = {'discrete': [], 'continuous': [], 'vlm_drift': []}

for step in range(200):
    ids = torch.randint(0, VOCAB, (B, T), device=device)
    targets = torch.randint(0, VOCAB, (B, T), device=device)   # synthetic teacher
    actions = torch.randn(B, H, D, device=device)
    noise = torch.randn_like(actions)
    t = torch.rand(B, device=device)

    logits, feats = vlm(ids)
    ctx = insulate(feats, enabled=True)
    a_t, target = flow_matching_target(actions, noise, t)
    v_pred = expert(a_t, t, ctx)

    out = ki_loss(logits, targets, v_pred, target, KIConfig())
    opt.zero_grad(set_to_none=True)
    out['loss'].backward()
    opt.step()

    history['discrete'].append(out['discrete'].item())
    history['continuous'].append(out['continuous'].item())
    drift = sum(((a - b)**2).sum().item() for a, b in zip(vlm.parameters(), vlm_init.parameters())) ** 0.5
    history['vlm_drift'].append(drift)

print(f'final discrete loss   : {history["discrete"][-1]:.3f}')
print(f'final continuous loss : {history["continuous"][-1]:.3f}')
print(f'final VLM drift (L2)  : {history["vlm_drift"][-1]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(history['discrete']);    axes[0].set(title='discrete (VLM) loss', xlabel='step')
axes[1].plot(history['continuous']);  axes[1].set(title='continuous (expert) loss', xlabel='step')
axes[2].plot(history['vlm_drift']);   axes[2].set(title='||VLM - VLM_init||₂', xlabel='step')
fig.suptitle('KI recipe — toy training run')
fig.tight_layout()

## C. Ablation: same loop with insulation off

Per the paper, removing the stop-gradient lets the continuous loss perturb the VLM's pretrained semantics. Here we just verify it changes the dynamics — the toy setting doesn't have rich enough semantics to show degradation.

In [ ]:
torch.manual_seed(0)
vlm2 = TinyVLM().to(device); expert2 = TinyExpert().to(device)
vlm2_init = copy.deepcopy(vlm2).requires_grad_(False)
opt2 = torch.optim.Adam(list(vlm2.parameters()) + list(expert2.parameters()), lr=3e-4)
history2 = {'discrete': [], 'continuous': [], 'vlm_drift': []}

for step in range(200):
    ids = torch.randint(0, VOCAB, (B, T), device=device)
    targets = torch.randint(0, VOCAB, (B, T), device=device)
    actions = torch.randn(B, H, D, device=device)
    noise = torch.randn_like(actions); t = torch.rand(B, device=device)
    logits, feats = vlm2(ids)
    ctx = insulate(feats, enabled=False)    # <-- KI OFF
    a_t, target = flow_matching_target(actions, noise, t)
    v_pred = expert2(a_t, t, ctx)
    out = ki_loss(logits, targets, v_pred, target, KIConfig())
    opt2.zero_grad(set_to_none=True); out['loss'].backward(); opt2.step()
    history2['discrete'].append(out['discrete'].item())
    history2['continuous'].append(out['continuous'].item())
    drift = sum(((a-b)**2).sum().item() for a, b in zip(vlm2.parameters(), vlm2_init.parameters())) ** 0.5
    history2['vlm_drift'].append(drift)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, key in zip(axes, ['discrete', 'continuous', 'vlm_drift']):
    ax.plot(history[key], label='KI on', lw=1.5)
    ax.plot(history2[key], '--', label='KI off', lw=1.5)
    ax.set(title=key, xlabel='step'); ax.legend()
fig.suptitle('KI on vs off (toy run)')
fig.tight_layout()

## Takeaways

- `insulate(features, enabled=True)` zeros out the gradient on the upstream net — verified numerically.
- The discrete loss still trains the VLM (drift > 0 in the KI-on run).
- KI-off lets the continuous loss perturb the VLM extra; in a real run, the paper shows this degrades language grounding.